# Celltypist Annotation R-Seurat Object

This notebook provides cluster characterization routines

# Analysis Description

Here the analysis is described. 

This analysis needs to be run in this singularity container, which also is available on dockerhub under [this link](link)
```{bash}
singularity shell --nv --bind /nemo:/nemo,/camp:/camp,/flask:/flask \
  /flask/apps/containers/all-singularity-images/r450.python310.ubuntu.22.04.v3.sif
```
and this python-venv environment
```{bash}
source /nemo/stp/babs/working/boeings/package_caches/venv/single_cell_venv_python310/bin/activate
```

The R-renv paths need to be set as follows:
```{bash}
export R_LIBS_USER=/nemo/stp/babs/working/boeings/package_caches/R/library/
export RENV_PATHS_LIBRARY=/nemo/stp/babs/working/boeings/package_caches/R/library/
export RENV_PATHS_CACHE=/nemo/stp/babs/working/boeings/package_caches/R/cache/
export RENV_PATHS_ROOT=/nemo/stp/babs/working/boeings/package_caches/renv
export PYTHONPYCACHEPREFIX=/nemo/stp/babs/working/boeings/package_caches/venv/pycache

```

In [ ]:
# Determine location
!hostname

In [ ]:
## Enable R

In [ ]:
# ── Active environment check ──────────────────────────────────────────────────
import sys, os

kernel_path = sys.executable
print(f'Active Python interpreter: {kernel_path}')
print()
print('>>> Please verify that this matches the kernel selected in your')
print('    Jupyter session (Kernel → Change Kernel…).')
print()
if 'envs' not in kernel_path and 'venv' not in kernel_path:
    print('WARNING: You appear to be running in the base environment.')
    print('         Activate your project venv/conda env and restart the kernel.')
else:
    print('OK: Running inside a virtual/conda environment.')

In [ ]:
import os
import sys
import ctypes

# 1. Force environment paths to point to your container's custom R 4.5.0
os.environ['R_HOME'] = '/usr/local/lib/R'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib/R/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

# 2. THE FIX: Force-load the container's libR.so into global memory scope
# This maps symbols like R_NegInf globally so matrixStats can see them
libR_path = "/usr/local/lib/R/lib/libR.so"
try:
    # Load with RTLD_GLOBAL to make symbols available to other shared objects
    ctypes.CDLL(libR_path, mode=ctypes.RTLD_GLOBAL)
    print("🚀 SUCCESS: Container libR.so pre-loaded with global symbol scope.")
except Exception as e:
    print(f"⚠️ Memory pre-load notice: {e}")

# 3. Force low-level rpy2 initialization
import rpy2.rinterface
try:
    rpy2.rinterface.initr()
except Exception as e:
    pass

# 4. Load the extension safely
%load_ext rpy2.ipython

# 5. Verify Active Memory Mapping Location
with open(f"/proc/{os.getpid()}/maps", "r") as f:
    r_libs = [line.split()[-1] for line in f.read().splitlines() if "libR.so" in line]

print("\n📊 --- Jupyter R Memory Check ---")
print(f"🔹 Loaded libR.so path: {list(set(r_libs))[0] if r_libs else 'None'}")
print("---------------------------------\n")

# Project Variables

In this section all relevant variables for this analysis are set. 

In [ ]:
import os
os.getcwd()

In [ ]:
## load object if it exists

In [ ]:
%%R
# Set the library path to your writable location
.libPaths()  # Verify it's set correctly

renv::restore()

In [ ]:
%%R
FN <- "../../../../workdir/SC25285.Seurat.Robj"
if (file.exists(FN)){
  load(FN)
} else {
  exit("No workspace found. Please run the previous step first and make sure it finished successfully.")
}

source("load.biologic.robj.R")


In [ ]:
%%R
sort(unique(OsC@meta.data$clusterName))

In [ ]:
%%R
OsC@meta.data['cta_leiden_selection'] <- OsC@meta.data$clusterName

In [ ]:
%%R
dim(OsC)

In [ ]:
%%R
if (!require("sceasy")){
  renv::install("cellgeni/sceasy")
}
renv::snapshot()

In [ ]:
%%R
reducs = names(OsC@reductions)
reducs = reducs[reducs %in% c("umap", "pca")]
reducs

In [ ]:
%%R -o outfile
Seurat::DefaultAssay(OsC) <- "RNA"

OsC_sub <- Seurat::DietSeurat(
  OsC,
  # counts = TRUE, # so, raw counts save to adata.layers['counts']
  layers = c("counts", "data"), # so, log1p counts save to adata.X when scale.data = False, else adata.layers['data']
  scale.data = FALSE, # if only scaled highly variable gene, the export to h5ad would fail. set to false
  features = rownames(OsC), # export all genes, not just top highly variable genes
  assays = "RNA",
  dimreducs = reducs,
  graphs = c("RNA_nn", "RNA_snn"), # to RNA_nn -> distances, RNA_snn -> connectivities
  misc = TRUE
)

# sce <- Seurat::as.SingleCellExperiment(OsCsub, assay = "RNA")
sce <- Seurat::as.SingleCellExperiment(OsC_sub)

#unlink("../../../../data/MYC.h5ad")

# Option A (faster)
outfile <- paste0("../../../../data/",Obio@parameterList$project_id,".conv.h5ad")

if (!require("sceasy")){
  renv::install("cellgeni/sceasy")
}


sceasy::convertFormat(
  sce, 
  from = "sce",
  to = "anndata",
  outFile = outfile
)

In [ ]:
outfile = str(outfile[0])
print(outfile)

In [ ]:
## Load into python

In [ ]:
import scanpy as sc

print(outfile)
adata =  sc.read_h5ad(outfile)
adata.shape


In [ ]:
adata.X[:10,:10].toarray()

In [ ]:
adata.layers['counts'] = adata.X.copy()
adata.layers['counts'].shape

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)


In [ ]:
import celltypist
from celltypist import models
import os

modSel = '../../../../scripts/SC25285/models/Healthy_Mouse_Liver.pkl'

os.path.isfile(modSel)


model = models.Model.load(model = modSel)

model.cell_types

In [ ]:
n_comps_required = 50  # Adjust this value as needed, but it needs to be 50 for celltypist to work.

# Rerun PCA with adjusted n_comps
sc.pp.pca(adata, n_comps=n_comps_required)
adata.obsm['X_pca'].shape
predictions = celltypist.annotate(adata, model = modSel , majority_voting = True)

predictions.predicted_labels

In [ ]:
# Now run predictions.to_adata()
adata = predictions.to_adata()

In [ ]:
adata.obs["clusterName"] = adata.obs["majority_voting"]

In [ ]:
## Quick check
import scanpy as sc

# Compute neighbors if not already done
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# Compute UMAP
sc.tl.umap(adata)
sc.pl.umap(adata, color = ['clusterName', 'meta_group'], legend_loc = 'on data')

In [ ]:
fn = "../../../../html_local/report_figures/umap.ct.annotation.pdf"
import matplotlib.pyplot as plt
plt.gcf().set_size_inches(4, 4)  # Set figure size (width, height) in inches

# Rasterize scatter points for smaller PDF size
for ax in plt.gcf().axes:
  for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig('../../../../html_local/report_figures/umap.annotation.ct.pdf', bbox_inches='tight', dpi=50)
plt.close()

In [ ]:
# Add cell IDs as a new column 'cellID' in adata.obs
import pandas as pd

#if 'cellID' not in adata.obs.columns:
#  adata.obs['cellID'] = adata.obs.index.astype(str)

colVec = ['cellID', 'clusterName','predicted_labels', 'conf_score']

df = adata.obs[colVec]


## Get HEX colors
cluster_colors = adata.uns['clusterName_colors']


# Convert RGB colors to hex
import matplotlib.colors as mcolors
import pandas as pd

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# Calculate UMAP
sc.tl.umap(adata)

# Plot the UMAP
sc.pl.umap(adata, color='clusterName')  # Replace 'louvain' with your desired metadata column

cluster_hex_colors = [mcolors.rgb2hex(color) for color in cluster_colors]


df_clusters = pd.DataFrame({
  'clusterName': adata.obs['clusterName'].cat.categories,
  'clusterColor': cluster_hex_colors
})



merged_df = pd.merge(df, df_clusters, on='clusterName')


fnOut = "../../../../data/celltypist.annotation.csv"
merged_df.to_csv(fnOut)

In [ ]:
%%R -i merged_df

# merged_df[merged_df$conf_score < 0.8, "clusterName"] <- "Unknown"

head(merged_df)

In [ ]:
%%R
df <- merged_df
df$X <- NULL
row.names(df) <- df$cellID
df$cellID <- NULL
df$over_clustering = NULL
head(df)

In [ ]:
%%R
unique(df$clusterName)

In [ ]:
%%R
df$clusterName <- gsub("Monocytes & Monocyte-derived cells", "MonocytesAndMCder", df$clusterName)
df$clusterName <- gsub("[.]", "", df$clusterName)
df$clusterName <- gsub(" ", "", df$clusterName)

In [ ]:
%%R
unique(df$clusterName)

In [ ]:
%%R
# Load necessary libraries
library(dplyr)
library(stringr)

# Define a function to clean column values
clean_column <- function(column) {
  column %>%
    str_replace_all(" ", "_") %>%  # Replace spaces with underscores
    str_replace_all("[^[:alnum:]_]", "")  # Remove non-standard characters
}

# Example usage with a data frame `df` and a column `columnName`
df <- df %>%
  mutate(clusterName = clean_column(clusterName)) %>%
  mutate(predicted_labels = clean_column(predicted_labels))



unique(df$clusterName)
df$conf_score <- round(df$conf_score, 3)

## Add color
#unique_clusters <- unique(df$clusterName)
#color_palette <- grDevices::rainbow(length(unique_clusters)) # Generate colors
#cluster_colors <- setNames(color_palette, unique_clusters)   # Name the colors by clusterName

# Add the clusterColor column to the data frame
#df <- df %>%
#mutate(clusterColor = cluster_colors[clusterName])

In [ ]:
%%R
head(df)

In [ ]:
%%R
OsC <- biologicToolsSC::addDf2seuratMetaData(obj = OsC, dfAdd = df)

In [ ]:
## Save in analysis workflow

In [ ]:
%%R
Obio@parameterList$clusterNameOrder <- sort(unique(OsC@meta.data$clusterName))

In [ ]:
%%R
Obio@parameterList$clusterNameOrder 

In [ ]:
%%R
source("save.biologic.robj.R")

In [ ]:
%%R
file = paste0(
             Obio@parameterList$localWorkDir,
             Obio@parameterList$project_id,
            ".Seurat.Robj")

file


In [ ]:
%%R
dim(OsC)

In [ ]:
%%R
save(OsC,
        file = paste0(
             Obio@parameterList$localWorkDir,
             Obio@parameterList$project_id,
            ".Seurat.Robj"
         )
   )

In [ ]:
%%R
unique(OsC@meta.data[,c("clusterName", "clusterColor")])

In [ ]:
%%R
dim(OsC)

In [ ]:
%%R
## Load into temp 
FN <- "../../../../workdir/temp/temp.workspace.RData"
if (file.exists(FN)){
    load(FN)

    ## Load new OsC
    file = paste0(
             Obio@parameterList$localWorkDir,
             Obio@parameterList$project_id,
            ".Seurat.Robj"
    )
    load(file)
    ## Load new Obio
    source("load.biologic.robj.R")

    ## Save new OsC
    tempDir <- "../../../../workdir/temp/"

    #if (!dir.exists(tempDir)){
    #  dir.create(tempDir, recursive = T)
    #}
    
    if (exists("whiteListWorkspace")){
        rm(list = setdiff(ls(), whiteListWorkspace))
    }
    
    FN <- "../../../../workdir/temp/temp.workspace.RData"
    save.image(FN)
    
} else {
    exit("No workspace found. Please run the previous step first and make sure it finished successfully.")
}

library(dplyr)